## Step 1. 原始 VCF 转换为 plink2 格式

这一步的目的是将原始压缩 VCF 转换为 plink2 的 `pgen/pvar/psam` 格式，便于后续做样本重命名、样本筛选、LD pruning 和 PCA。

输入文件：
- 原始 VCF：`CIMA_phasing.merged.vcf.gz`

输出文件：
- `CIMA_raw.pgen`
- `CIMA_raw.pvar`
- `CIMA_raw.psam`

说明：
- 这一阶段仍保留原始样本名
- 暂时不做 PCA，不做 GWAS
- 先把基因型数据整理成可操作格式

In [6]:
from pathlib import Path

csv_path = Path("/mnt/zzbnew/peixunban/zhanghebin/CIMA_multiomics_regulation/data/raw/CIMA/Metabolites_and_Lipids/CIMA_Sample_Plasma_Metabolites_and_Lipids_Results.csv")

print("exists:", csv_path.exists())
print("is_file:", csv_path.is_file())
print("size_MB:", round(csv_path.stat().st_size / 1024 / 1024, 2) if csv_path.exists() else None)

exists: True
is_file: True
size_MB: 5.39


In [7]:
import pandas as pd

df = pd.read_csv(csv_path)
print(df.shape)
display(df.head())
print(df.columns.tolist())

(390, 1550)


,sample,O-Phosphoryl-ethanolamine,Acesulfame,Acetic acid,Propanoic acid,Glycolic acid,But-2-enoic acid,Valeric acid,2-Aminoisobutyric acid,L-Alpha-aminobutyric acid,...,PI 20:0-22:6,PI 20:5-22:0,PI 20:4-22:1,PI 20:3-22:2,PI 20:2-22:3,PI 20:1-22:4,PI 20:0-22:5,PI 20:4-22:0,PI 20:3-22:1,PI 20:2-22:2
0,CIMA_H015,1.225213,0.032842,3.455972,4.275737,20.229021,0.062537,0.634065,8.593622,22.512360,...,NaN,11.825791,NaN,15.923843,11.58539,NaN,11.837008,18.987968,30.148115,NaN
1,CIMA_H016,1.272470,0.086072,2.996980,3.598019,18.835006,0.039117,0.566319,9.118566,21.378370,...,NaN,11.376650,24.858793,17.906786,NaN,NaN,NaN,NaN,29.881211,NaN
2,CIMA_H017,1.172774,0.126888,7.860442,3.267677,22.291050,0.055000,0.688793,5.751301,28.211497,...,NaN,NaN,21.264320,12.559251,NaN,NaN,NaN,11.276070,NaN,NaN
3,CIMA_H018,1.618271,0.053564,4.512696,1.991009,15.116417,0.074073,1.124161,5.912626,34.551486,...,11.904203,NaN,38.426573,NaN,NaN,NaN,11.727066,11.178847,NaN,NaN
4,CIMA_H019,1.755173,0.071668,2.400409,4.733403,21.703553,0.094791,0.926420,13.669501,33.472662,...,NaN,NaN,NaN,11.825061,NaN,NaN,NaN,NaN,34.482757,15.030102


['sample', 'O-Phosphoryl-ethanolamine', 'Acesulfame', 'Acetic acid', 'Propanoic acid', 'Glycolic acid', 'But-2-enoic acid', 'Valeric acid', '2-Aminoisobutyric acid', 'L-Alpha-aminobutyric acid', 'alpha-Hydroxyisobutyric acid', '2-Hydroxybutyric acid', '3-Hydroxybutyric acid', 'Acetylglycine', '3-Hydroxyisovaleric acid', '2-Hydroxy-2-methylbutyric acid', '5-Aminoimidazole-4-carboxamide', 'L-Pipecolic acid', '4-Hydroxyproline', 'cis-4-Hydroxy-D-proline', 'Hydroxyisocaproic acid', 'Erythronic acid', '2-Methylbenzoic acid', 'N-Methylnicotinamide', '3,4-Dihydroxybenzaldehyde', '2-Ethylhexanoic acid', '2-Methylheptanoic Acid', '4-Acetamidobutanoic acid', '4-Guanidinobutyric acid', 'N,N-Dimethyl-L-Valine', 'Monomethyl glutaric acid', '3-Methyloxindole', '3,4-Dihydro-2H-1-benzopyran-2-one', 'D-Xylose', 'Hydrocinnamic acid', '2-HYDROXY-4-(METHYLTHIO)BUTANOATE', 'Phenylglyoxylic acid', 'D-Arabinose', '2-Phenylpropionic acid', '2-Phenylglycine', '(S)C(S)S-S-Methylcysteine sulfoxide', '3-Hydroxyph

In [8]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "missing_rate": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values(["missing_rate", "n_unique"], ascending=[False, True])

display(summary)

,dtype,n_missing,missing_rate,n_unique
PG 18:3-22:3,float64,376,96.41,13
PS 20:0-20:0,float64,374,95.90,15
PG 18:4-20:0,float64,370,94.87,17
PG 18:4-20:1,float64,369,94.62,17
PS 20:4-20:4,float64,369,94.62,20
...,...,...,...,...
TAG54:0-FA16:0,float64,0,0.00,390
TAG56:10-FA18:2,float64,0,0.00,390
TAG56:2-FA20:0,float64,0,0.00,390
TAG56:2-FA16:0,float64,0,0.00,390


In [9]:
for i, c in enumerate(df.columns):
    print(i, repr(c))

0 'sample'
1 'O-Phosphoryl-ethanolamine'
2 'Acesulfame'
3 'Acetic acid'
4 'Propanoic acid'
5 'Glycolic acid'
6 'But-2-enoic acid'
7 'Valeric acid'
8 '2-Aminoisobutyric acid'
9 'L-Alpha-aminobutyric acid'
10 'alpha-Hydroxyisobutyric acid'
11 '2-Hydroxybutyric acid'
12 '3-Hydroxybutyric acid'
13 'Acetylglycine'
14 '3-Hydroxyisovaleric acid'
15 '2-Hydroxy-2-methylbutyric acid'
16 '5-Aminoimidazole-4-carboxamide'
17 'L-Pipecolic acid'
18 '4-Hydroxyproline'
19 'cis-4-Hydroxy-D-proline'
20 'Hydroxyisocaproic acid'
21 'Erythronic acid'
22 '2-Methylbenzoic acid'
23 'N-Methylnicotinamide'
24 '3,4-Dihydroxybenzaldehyde'
25 '2-Ethylhexanoic acid'
26 '2-Methylheptanoic Acid'
27 '4-Acetamidobutanoic acid'
28 '4-Guanidinobutyric acid'
29 'N,N-Dimethyl-L-Valine'
30 'Monomethyl glutaric acid'
31 '3-Methyloxindole'
32 '3,4-Dihydro-2H-1-benzopyran-2-one'
33 'D-Xylose'
34 'Hydrocinnamic acid'
35 '2-HYDROXY-4-(METHYLTHIO)BUTANOATE'
36 'Phenylglyoxylic acid'
37 'D-Arabinose'
38 '2-Phenylpropionic acid'
39 